<a href="https://colab.research.google.com/github/victorbaraunaAcad/crise-saude-ufam/blob/Leonardo-branch/coleta.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import json, time, datetime, pathlib, unicodedata
from getpass import getpass

import requests
import pandas as pd

print(f"pandas  {pd.__version__}")
print(f"requests {requests.__version__}")

pandas  2.2.3
requests 2.32.4


In [ ]:
USUARIO = "victorbaraunaAcad"      # trocar pelo seu usuário
REPO    = "crise-saude-ufam" # trocar pelo nome do repositório
'''
token = getpass("Cole o token do GitHub (fine-grained, Contents: Read/write): ")

!git clone https://{token}@github.com/{USUARIO}/{REPO}.git
%cd {REPO}
!git config user.name "victorbaraunaAcad"
!git config user.email "victor.barauna@icomp.ufam.edu.br"
'''
RAIZ     = pathlib.Path.cwd()
BRUTOS   = RAIZ / "dados_brutos"
TRATADOS = RAIZ / "dados_tratados"
BRUTOS.mkdir(exist_ok=True); TRATADOS.mkdir(exist_ok=True)
print("Trabalhando em:", RAIZ)

Cole o token do GitHub (fine-grained, Contents: Read/write): ··········
Cloning into 'crise-saude-ufam'...
remote: Enumerating objects: 36, done.
remote: Counting objects: 100% (36/36), done.
remote: Compressing objects: 100% (29/29), done.
remote: Total 36 (delta 16), reused 12 (delta 5), pack-reused 0 (from 0)
Receiving objects: 100% (36/36), 39.89 KiB | 3.07 MiB/s, done.
Resolving deltas: 100% (16/16), done.
/content/crise-saude-ufam
Trabalhando em: /content/crise-saude-ufam


In [ ]:
ARQ_PROV = RAIZ / "proveniencia.csv"

def registrar(fonte, url, metodo, parametros, n_linhas, arquivo_bruto):
    linha = pd.DataFrame([{
        "fonte": fonte, "url": url, "metodo": metodo,
        "parametros": json.dumps(parametros, ensure_ascii=False),
        "n_linhas": n_linhas,
        "arquivo_bruto": str(pathlib.Path(arquivo_bruto).relative_to(RAIZ)),
        "coletado_em": datetime.datetime.now().astimezone().isoformat(timespec="seconds"),
    }])
    cabecalho = not ARQ_PROV.exists() or ARQ_PROV.stat().st_size == 0
    linha.to_csv(ARQ_PROV, mode="a", header=cabecalho, index=False)
    print(f"  ✓ proveniência: {fonte} ({n_linhas} linhas)")

def normalizar(s):
    s = unicodedata.normalize("NFKD", str(s)).encode("ascii", "ignore").decode()
    return s.upper().strip()

In [ ]:
CODIGOS_UF = {"AM": 13, "RS": 43}
CIDADES_CENTRO_OESTE = {"Cuiabá": 51, "Goiânia": 52, "Brasília": 53}

def buscar_municipios_uf(cod_uf, sigla):
    url = f"https://servicodados.ibge.gov.br/api/v1/localidades/estados/{cod_uf}/municipios"
    r = requests.get(url, timeout=30)
    r.raise_for_status()
    bruto = r.json()

    arq = BRUTOS / f"ibge_localidades_{sigla}.json"
    arq.write_text(json.dumps(bruto, ensure_ascii=False, indent=2), encoding="utf-8")
    registrar(f"IBGE Localidades — {sigla}", url, "API REST (GET, JSON)",
              {"uf": cod_uf}, len(bruto), arq)

    return pd.json_normalize(bruto).rename(columns={
        "id": "codigo_ibge", "nome": "nome_municipio",
        "microrregiao.mesorregiao.UF.sigla": "uf",
    })[["codigo_ibge", "nome_municipio", "uf"]]

partes = []

# AM e RS inteiros
for sigla, cod_uf in CODIGOS_UF.items():
    partes.append(buscar_municipios_uf(cod_uf, sigla))

# Cuiabá, Goiânia, Brasília — filtradas de dentro do estado de cada uma
for cidade, cod_uf in CIDADES_CENTRO_OESTE.items():
    df_uf = buscar_municipios_uf(cod_uf, f"CO_{cod_uf}")
    df_uf["nome_norm"] = df_uf["nome_municipio"].map(normalizar)
    achado = df_uf[df_uf["nome_norm"] == normalizar(cidade)]
    if achado.empty:
        print(f"  ✗ não achei {cidade} — confira o nome retornado pela API")
    partes.append(achado.drop(columns="nome_norm"))

municipios = pd.concat(partes, ignore_index=True).drop_duplicates("codigo_ibge")
municipios["codigo_ibge"] = municipios["codigo_ibge"].astype("int64")
municipios["nome_norm"] = municipios["nome_municipio"].map(normalizar)

print(municipios.groupby("uf").size())
municipios.head()

  ✓ proveniência: IBGE Localidades — AM (62 linhas)
  ✓ proveniência: IBGE Localidades — RS (497 linhas)
  ✓ proveniência: IBGE Localidades — CO_51 (142 linhas)
  ✓ proveniência: IBGE Localidades — CO_52 (246 linhas)
  ✓ proveniência: IBGE Localidades — CO_53 (1 linhas)
uf
AM     62
DF      1
GO      1
MT      1
RS    497
dtype: int64


,codigo_ibge,nome_municipio,uf,nome_norm
0,1300029,Alvarães,AM,ALVARAES
1,1300060,Amaturá,AM,AMATURA
2,1300086,Anamã,AM,ANAMA
3,1300102,Anori,AM,ANORI
4,1300144,Apuí,AM,APUI


In [ ]:
print("Duplicatas:", municipios["codigo_ibge"].duplicated().sum())      # precisa ser 0
print("Total de linhas:", len(municipios))                              # esperado: ~62+497+3 = 562
print("Dígitos do código:", municipios["codigo_ibge"].astype(str).str.len().unique())  # precisa ser [7]
municipios[municipios["uf"].isin(["MT","GO","DF"])]                     # confirma as 3 cidades certas

Duplicatas: 0
Total de linhas: 562
Dígitos do código: [7]


,codigo_ibge,nome_municipio,uf,nome_norm
559,5103403,Cuiabá,MT,CUIABA
560,5208707,Goiânia,GO,GOIANIA
561,5300108,Brasília,DF,BRASILIA


In [ ]:
'''
!git add -A
!git commit -m "Fonte 1: municipios AM inteiro + RS inteiro + Cuiaba/Goiania/Brasilia"
!git push
'''

[main 3659e2b] Fonte 1: municipios AM inteiro + RS inteiro + Cuiaba/Goiania/Brasilia
 1 file changed, 5 insertions(+)
Enumerating objects: 5, done.
Counting objects: 100% (5/5), done.
Delta compression using up to 2 threads
Compressing objects: 100% (3/3), done.
Writing objects: 100% (3/3), 386 bytes | 386.00 KiB/s, done.
Total 3 (delta 2), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (2/2), completed with 2 local objects.
To https://github.com/victorbaraunaAcad/crise-saude-ufam.git
   ea0c9f3..3659e2b  main -> main


In [ ]:
'''
!git config pull.rebase false
!git pull origin main --no-edit
!git push

From https://github.com/victorbaraunaAcad/crise-saude-ufam
 * branch            main       -> FETCH_HEAD
Already up to date.
Everything up-to-date


In [ ]:
UFS_PROJETO = {"AM": 13, "RS": 43, "MT": 51, "GO": 52, "DF": 53}
ufs_ids = ",".join(str(c) for c in UFS_PROJETO.values())

# candidatos de formato de período — testamos até um funcionar
candidatos_periodo = [
    "202301-202404",   # AAAATT (ano+trimestre) intervalo
    "202301|202302|202303|202304|202401|202402|202403|202404",  # lista explícita
    "all",             # todos os períodos disponíveis
]

url_base = f"https://servicodados.ibge.gov.br/api/v3/agregados/4099/periodos/{{}}/variaveis/4099?localidades=N3[{ufs_ids}]"

for periodo in candidatos_periodo:
    url = url_base.format(periodo)
    r = requests.get(url, timeout=30)
    print(f"período='{periodo}' → status {r.status_code}")
    if r.status_code == 200:
        print("   FUNCIONOU — usar este formato")
        break
    else:
        print("   ", r.text[:200])

período='202301-202404' → status 200
   FUNCIONOU — usar este formato


In [ ]:
PERIODO_OK = "202301-202404"  # <- trocar pelo que funcionou na Célula 6

url_desemprego = url_base.format(PERIODO_OK)
r = requests.get(url_desemprego, timeout=30)
r.raise_for_status()
bruto = r.json()

arq = BRUTOS / "ibge_desemprego_ufs.json"
arq.write_text(json.dumps(bruto, ensure_ascii=False, indent=2), encoding="utf-8")
registrar("IBGE SIDRA — Taxa de desocupação (tab. 4099)", r.url,
          "API REST (GET, JSON)", {"agregado": 4099, "periodo": PERIODO_OK},
          len(bruto), arq)

linhas = []
for item in bruto:
    for res in item["resultados"]:
        for serie in res["series"]:
            for periodo, valor in serie["serie"].items():
                linhas.append({
                    "uf_codigo": int(serie["localidade"]["id"]),
                    "uf_nome": serie["localidade"]["nome"],
                    "periodo_trimestre": periodo,
                    "taxa_desocupacao": pd.to_numeric(valor, errors="coerce"),
                })

desemprego_uf = pd.DataFrame(linhas)
print(desemprego_uf.shape)
desemprego_uf.head(10)

  ✓ proveniência: IBGE SIDRA — Taxa de desocupação (tab. 4099) (1 linhas)
(40, 4)


,uf_codigo,uf_nome,periodo_trimestre,taxa_desocupacao
0,13,Amazonas,202301,10.5
1,13,Amazonas,202302,9.7
2,13,Amazonas,202303,9.6
3,13,Amazonas,202304,8.8
4,13,Amazonas,202401,9.8
5,13,Amazonas,202402,8.0
6,13,Amazonas,202403,8.2
7,13,Amazonas,202404,8.3
8,43,Rio Grande do Sul,202301,5.4
9,43,Rio Grande do Sul,202302,5.3


In [ ]:
print("UFs presentes:", sorted(desemprego_uf["uf_codigo"].unique()))   # espera [13,43,51,52,53]
print("Ausentes:", desemprego_uf["taxa_desocupacao"].isnull().sum())
desemprego_uf.groupby("uf_nome")["taxa_desocupacao"].agg(["min","max","mean"])

UFs presentes: [np.int64(13), np.int64(43), np.int64(51), np.int64(52), np.int64(53)]
Ausentes: 0


,min,max,mean
uf_nome,,,
Amazonas,8.0,10.5,9.1125
Distrito Federal,8.8,12.0,9.5875
Goiás,4.9,6.8,5.7500
Mato Grosso,2.3,4.5,3.2250
Rio Grande do Sul,4.5,5.9,5.3250


In [ ]:
'''
!git add -A
!git commit -m "Fonte 2: taxa de desocupacao por UF (PNAD Continua, tab 4099)"
!git push

[main 96e85fe] Fonte 2: taxa de desocupacao por UF (PNAD Continua, tab 4099)
 2 files changed, 115 insertions(+)
 create mode 100644 dados_brutos/ibge_desemprego_ufs.json
Enumerating objects: 8, done.
Counting objects: 100% (8/8), done.
Delta compression using up to 2 threads
Compressing objects: 100% (5/5), done.
Writing objects: 100% (5/5), 1.13 KiB | 1.13 MiB/s, done.
Total 5 (delta 3), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (3/3), completed with 3 local objects.
To https://github.com/victorbaraunaAcad/crise-saude-ufam.git
   3659e2b..96e85fe  main -> main


In [ ]:
cod_mun = ",".join(municipios["codigo_ibge"].astype(str))

FONTES_PERFIL = {
    "populacao": "https://servicodados.ibge.gov.br/api/v3/agregados/6579/periodos/-1/variaveis/9324",
    "pib_percapita": "https://servicodados.ibge.gov.br/api/v3/agregados/5938/periodos/-1/variaveis/593",
}

for nome, url_base in FONTES_PERFIL.items():
    url = f"{url_base}?localidades=N6[{cod_mun}]"
    r = requests.get(url, timeout=60)
    print(nome, r.status_code)
    if r.status_code != 200:
        print("  ", r.text[:200])
        continue
    bruto = r.json()
    arq = BRUTOS / f"ibge_{nome}_municipios.json"
    arq.write_text(json.dumps(bruto, ensure_ascii=False, indent=2), encoding="utf-8")
    registrar(f"IBGE SIDRA — {nome}", url, "API REST (GET, JSON)", {}, len(bruto), arq)

populacao 500
   {"statusCode":500,"message":"Internal server error"}
pib_percapita 500
   {"statusCode":500,"message":"Internal server error"}


In [ ]:
UFS_INTEIRAS = [13, 43]   # AM, RS — todos os municípios dessas UFs
codigos_co = municipios[municipios["uf"].isin(["MT","GO","DF"])]["codigo_ibge"].tolist()  # as 3 cidades

for nome, url_base in FONTES_PERFIL.items():
    partes_bruto = []

    # 1) AM e RS inteiros, via filtro por UF
    localidades_uf = f"N6[N3[{','.join(str(u) for u in UFS_INTEIRAS)}]]"
    url1 = f"{url_base}?localidades={localidades_uf}"
    r1 = requests.get(url1, timeout=60)
    print(f"{nome} (UFs inteiras):", r1.status_code)
    if r1.status_code == 200:
        partes_bruto.append(("uf_inteira", url1, r1.json()))
    else:
        print("  ", r1.text[:200])

    # 2) as 3 cidades do Centro-Oeste, via lista explícita curta
    localidades_co = f"N6[{','.join(str(c) for c in codigos_co)}]"
    url2 = f"{url_base}?localidades={localidades_co}"
    r2 = requests.get(url2, timeout=60)
    print(f"{nome} (Centro-Oeste):", r2.status_code)
    if r2.status_code == 200:
        partes_bruto.append(("centro_oeste", url2, r2.json()))
    else:
        print("  ", r2.text[:200])

    # salva cada resposta como um bruto separado (preserva exatamente como veio)
    for tag, url, bruto in partes_bruto:
        arq = BRUTOS / f"ibge_{nome}_{tag}.json"
        arq.write_text(json.dumps(bruto, ensure_ascii=False, indent=2), encoding="utf-8")
        registrar(f"IBGE SIDRA — {nome} ({tag})", url, "API REST (GET, JSON)", {}, len(bruto), arq)

populacao (UFs inteiras): 200
populacao (Centro-Oeste): 200
  ✓ proveniência: IBGE SIDRA — populacao (uf_inteira) (1 linhas)
  ✓ proveniência: IBGE SIDRA — populacao (centro_oeste) (1 linhas)
pib_percapita (UFs inteiras): 500
   {"statusCode":500,"message":"Internal server error"}
pib_percapita (Centro-Oeste): 500
   {"statusCode":500,"message":"Internal server error"}


In [ ]:
url_meta = "https://servicodados.ibge.gov.br/api/v3/agregados/5938/metadados"
r = requests.get(url_meta, timeout=30)
print(r.status_code)
meta = r.json()

print("Nome da tabela:", meta.get("nome"))
print("\nVariáveis disponíveis:")
for v in meta.get("variaveis", []):
    print(f"  id={v['id']}  nome={v['nome']}  unidade={v.get('unidade')}")

print("\nNíveis territoriais aceitos:", meta.get("nivelTerritorial"))
print("\nPeríodos disponíveis (últimos 5):", meta.get("periodicidade"))

200
Nome da tabela: Produto interno bruto a preços correntes, impostos, líquidos de subsídios, sobre produtos a preços correntes e valor adicionado bruto a preços correntes total e por atividade econômica, e respectivas participações - Referência 2010

Variáveis disponíveis:
  id=37  nome=Produto Interno Bruto a preços correntes  unidade=Mil Reais
  id=553  nome=Participação do produto interno bruto a preços correntes no produto interno bruto a preços correntes da microrregião geográfica  unidade=%
  id=552  nome=Participação do produto interno bruto a preços correntes no produto interno bruto a preços correntes da mesorregião geográfica  unidade=%
  id=497  nome=Participação do produto interno bruto a preços correntes no produto interno bruto a preços correntes da unidade da federação  unidade=%
  id=530  nome=Participação do produto interno bruto a preços correntes no produto interno bruto a preços correntes da grande região  unidade=%
  id=496  nome=Participação do produto interno b

In [ ]:
# 1) achar automaticamente a variável de "per capita" na lista que já veio
for v in meta.get("variaveis", []):
    if "per capita" in v["nome"].lower():
        print(f"id={v['id']}  nome={v['nome']}  unidade={v.get('unidade')}")

In [ ]:
VAR_PIB = 37   # <- trocar pelo id que a busca acima encontrar

url1 = f"https://servicodados.ibge.gov.br/api/v3/agregados/5938/periodos/2021/variaveis/{VAR_PIB}?localidades=N6[N3[13,43]]"
r1 = requests.get(url1, timeout=60)
print("PIB (UFs inteiras):", r1.status_code)
if r1.status_code != 200:
    print("  ", r1.text[:300])

url2 = f"https://servicodados.ibge.gov.br/api/v3/agregados/5938/periodos/2021/variaveis/{VAR_PIB}?localidades=N6[{','.join(str(c) for c in codigos_co)}]"
r2 = requests.get(url2, timeout=60)
print("PIB (Centro-Oeste):", r2.status_code)
if r2.status_code != 200:
    print("  ", r2.text[:300])

PIB (UFs inteiras): 200
PIB (Centro-Oeste): 200


In [ ]:
for tag, url, resp in [("uf_inteira", url1, r1), ("centro_oeste", url2, r2)]:
    bruto = resp.json()
    arq = BRUTOS / f"ibge_pib_total_{tag}.json"
    arq.write_text(json.dumps(bruto, ensure_ascii=False, indent=2), encoding="utf-8")
    registrar(f"IBGE SIDRA — PIB total municipal ({tag})", url,
              "API REST (GET, JSON)", {"agregado": 5938, "variavel": VAR_PIB, "periodo": 2021},
              len(bruto), arq)

  ✓ proveniência: IBGE SIDRA — PIB total municipal (uf_inteira) (1 linhas)
  ✓ proveniência: IBGE SIDRA — PIB total municipal (centro_oeste) (1 linhas)


In [ ]:
prov = pd.read_csv("proveniencia.csv")
prov = prov.drop_duplicates(subset=["fonte","url","parametros"], keep="first")
prov.to_csv("proveniencia.csv", index=False)

In [ ]:
'''
!git add -A
!git commit -m "Fonte 3: PIB total municipal (tab 5938, var 37, ano 2021) + populacao"
!git push